# Preconditioning interface for instationary control problems

We now show how to build preconditioners for instationary control problems. Due to the complexity of the matrix systems, we will be applying the identity as a preconditioner.

## Preconditioning heat control problems

Given $\beta>0$, $\Omega \subset \mathrm{R}^d$, with $d \in \{1,2,3\}$, and $t_f>0$, we consider the solution of the following heat control problem:

$$
    \min_{v, u} ~ \frac{1}{2} \int_0^{t_f} \| v - v_d
		\|^2 \mathrm{d} t + \frac{\beta}{2} \int_0^{t_f} 
		\| u \|^2 \mathrm{d} t
$$

subject to

$$
    \frac{\partial v}{\partial t} - \nabla^2 v = f + u \qquad \mathrm{in} \;
					\Omega \times (0, t_f),
$$
$$
    v(\mathbf{x},t) = g(\mathbf{x},t) \qquad \mathrm{on} \; \partial \Omega,
$$
$$
    v(\mathbf{x},0) = v_0(\mathbf{x}) \qquad \mathrm{in} \; \Omega.
$$

We integrate the problem in $\Omega :=[-1,1]^2$ up to $t_f=2$, and set $v = v_d := \exp(t_f - t) \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2}) + 1$ and $u=0$, and derive the force funtion $f$, the boundary conditions $g$, and the initial condition $v_0(\mathbf{x})$ accordingly. For this example, we set $\beta = 10^{-4}$.

In [ ]:
from firedrake import *
from control.control import Instationary

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

beta = 1.0e-4
t_f = 2.0
n_t = 5

space_0 = FunctionSpace(mesh, "Lagrange", 1)


def forw_diff_operator(trial, test, v, t):
    # spatial differential for the forward problem
    return inner(grad(trial), grad(test)) * dx


def desired_state(test, t):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)

    # desired state
    v_d = Function(space)
    v_d.interpolate(exp(t_f - t) * cos(0.5 * pi * X[0]) * cos(0.5 * pi * X[1]) + 1.0)

    return inner(v_d, test) * dx, v_d


def force_f(test, t):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)

    c = Constant(exp(t_f - t) * (-1.0 + 0.5 * pi * pi))

    # force function
    f = Function(space)
    f.interpolate(c * cos(0.5 * pi * X[0]) * cos(0.5 * pi * X[1]))

    return inner(f, test) * dx


def initial_condition(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)

    # desired state
    v_0 = Function(space)
    v_0.interpolate(exp(t_f) * cos(0.5 * pi * X[0]) * cos(0.5 * pi * X[1]) + 1.0)

    return v_0


def bcs_v_t(space_0, t):
    bcs_v_t = DirichletBC(space_0, 1.0, "on_boundary")
    return bcs_v_t


heat_control = Instationary(
    space_0, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, beta=beta, n_t=n_t,
    initial_condition=initial_condition,
    time_interval=(0.0, t_f), bcs_v=bcs_v_t)

We now construct the preconditioner. This has to take as input the control object, the $(1,2)$- and the $(2,1)$-blocks, the full space-time space for the solutions $v$ and $\zeta$, and the corresponding boundary conditions. The $(1,1)$- and the $(2,2)$-blocks can be constructed from the form sel._M_v, while the time step can be recovered easily. As we mentioned, since this notebook is just to explaining how to build a preconditioner, we will be employing and identity as preconditioner, meaning that no preconditioner is applied. This is not recommended, as the preconditioned Krylov method will probably not converge to the given tolerance.

In [ ]:
def P(self, block_01, block_10,
      full_space_v_help, bcs_v, bcs_zeta):
    # definition of preconditioner
    def pc_linear(u_0, u_1, b_0, b_1):
        u_0.assign(b_0.riesz_representation("l2"))
        u_1.assign(b_1.riesz_representation("l2"))

    return pc_linear

We can now solve the problem by passing the preconditioner as kwarg to the module linear_solve.

In [ ]:
heat_control.linear_solve(P=P, outputs=False, plots=False)

In case of non-linear control problems, the preconditioner can be given as input to the module non_linear_solve(), by passing it to the kwarg P.

In a similar way, for incompressible control problems, either in the linear or non-linear setting, one has to give as input to the corresponding solver a function for the construction of the preconditioner to the kwarg P. In this case, the function should have the following form:

In [ ]:
def P(self, block_00_int, block_01_int,
      block_10_int, block_11_int, B,
      full_nullspace_v, full_nullspace_zeta,
      bcs_v, bcs_zeta):
    # definition of preconditioner
    def pc_linear(u_0, u_1, b_0, b_1):
        u_0.assign(b_0.riesz_representation("l2"))
        u_1.assign(b_1.riesz_representation("l2"))

    return pc_linear

In the previous function, the input block_00_int, block_01_int, block_10_int, and block_11_int contain the dictionary of the inner blocks coupling the state and adjoint velocities; further, full_nullspace_v and full_nullspace_zeta contain the nullspaces for the two variables; finally, bcs_v and bcs_zeta contain the homogenized boundary conditions on $v$ and $\zeta$. This has to be built in this way in case the user would like to apply an inner solver with the class MultiBlockSystem for the system of the coupled velocities.